# Project 1: modern conical-flow CFD in Python

**Ehsan Roohi · historical Fortran coursework revisited in 2026**

One self-contained notebook includes the numerical source modules, independent verification,
two wall-matched viscous seeds and controls for all implemented fluxes. It does not call a packaged CFD solver.

[Source and migration guide](https://github.com/Ehsan-Roohi/Ehsan-Roohi/tree/main/courses/cfd-2/modern-python/conical) ·
[Published result report](https://github.com/Ehsan-Roohi/Ehsan-Roohi/blob/main/courses/cfd-2/modern-python/conical/RESULTS.md) ·
[DG companion](https://github.com/Ehsan-Roohi/Ehsan-Roohi/blob/main/courses/cfd-2/modern-python/conical/DG.md)

| Choice | Available methods |
|---|---|
| Euler face flux | Roe, Van Leer, HLLC, HLLE, Rusanov, AUSM, AUSM+, AUSM+-up, AUSM+-up2, SLAU2, EC+LF |
| Finite-volume reconstruction | Constant, MUSCL-MC, CWENO3/5, characteristic CWENO3/5 |
| Viscous reconstruction | Primitive-minmod and nonuniform CWENO variants |
| Viscous transport | Sutherland/constant viscosity in the source; heat conduction; adiabatic or isothermal no-slip wall; gradients of order 2/4 |
| Optional companion | Inviscid angular modal DG, degrees 0/1/2; shared Euler fluxes |

DG is a separate high-order discretization, not another Riemann flux. The FR connection is checked for linear advection with Radau correction functions; a complete FR cone solver is not claimed.

**Scope:** the cone model is an angular reduction. Viscous results describe a local radial station,
not downstream boundary-layer growth or full axisymmetric Navier–Stokes validation. Some high-order/shock runs do not converge.
The report preserves those failures. A plotted curve alone is not an accuracy certificate.

Choose **Runtime → Run all** for the default finite-volume example. The first execution unpacks the embedded sources.
The default CPU example is Mach 2.35, 10° cone, HLLC/CWENO3 on 60 cells; runtime depends on the Colab host.


In [ ]:
# Standard scientific Python only; Colab normally includes both dependencies.
import importlib.util, subprocess, sys
missing = [name for name in ("numpy", "matplotlib") if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
import numpy as np
import matplotlib
print("Python:", sys.version.split()[0], "NumPy:", np.__version__, "Matplotlib:", matplotlib.__version__)


In [ ]:
# Embedded, hashed source bundle. No GitHub download or credentials are required.
import base64, hashlib, io, json, os, zipfile
from pathlib import Path
BUNDLE_SHA256 = 'ad15cdf205c76c48e474c0498d7c98975b4a7d39e5fed8a0aeb39508ca283dba'
BUNDLE_BASE64 = '...'
bundle_bytes = base64.b64decode(BUNDLE_BASE64)
assert hashlib.sha256(bundle_bytes).hexdigest() == BUNDLE_SHA256
WORK = (Path("/content") if Path("/content").exists() else Path.cwd()) / "conical-modern"
WORK.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(bundle_bytes)) as z:
    for name in z.namelist():
        destination = (WORK / name).resolve()
        if not destination.is_relative_to(WORK.resolve()):
            raise ValueError("Unsafe archive path")
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(z.read(name))
manifest = json.loads((WORK / "EMBEDDED-MANIFEST.json").read_text())
assert all(hashlib.sha256((WORK/name).read_bytes()).hexdigest() == digest for name,digest in manifest.items())
sys.path.insert(0, str(WORK))
from colab_api import run_case, plot_comparison
from flux import FLUXES, numerical_flux
from state import freestream, physical_flux
print("Embedded files verified:", len(manifest))
print("All implemented fluxes:", ", ".join(FLUXES))


## Independent checks

The quick check tests equal-state consistency for every flux and reruns DG conservation,
degree-zero reduction, smooth advection order, mean-preserving limiting and linear FR/DG equivalence.
The published finite-volume verification also covers exact Sod flow, nonuniform polynomial operators,
viscous constitutive laws and Couette viscous heating. Enable the optional extended check to rerun it.


In [ ]:
q = freestream(np.array([.3, .4]), 2.35)
for name in FLUXES:
    error = np.max(abs(numerical_flux(name, q, q, 1.4, mach_ref=2.35)-physical_flux(q)))
    assert error < 1e-12, (name, error)
from verify_dg import verify
dg_verification = verify()
assert dg_verification["passed"]
print("All 11 flux consistency checks and DG verification passed.")
print("DG measured orders:", {p: rows[-1]["order"] for p,rows in dg_verification["periodic_advection_order"].items()})
RUN_EXTENDED_VERIFICATION = False # @param {type:"boolean"}
if RUN_EXTENDED_VERIFICATION:
    subprocess.check_call([sys.executable, str(WORK / "verify_extension.py")])
    subprocess.check_call([sys.executable, str(WORK / "shock_tube.py")])


## Choose a case and run it

With `Use_default_physics_case=True`, inviscid and DG runs use Mach 2.35/outer 45°;
viscous runs use Mach 7.95/outer 22°, primitive-minmod and the requested gradient order.
Disable it to use your explicit Mach/outer/reconstruction settings.

The viscous defaults start from a published 60-cell seed only if the physical parameters,
thermal wall and grid match. Changing the flux or operator still solves the selected equations.
Seeded iteration counts are not fair speed rankings. If the seed cannot be used, the solver starts from inviscid cell averages.

`Compare_all_fluxes=True` runs all 11 on the same physical configuration. This is more expensive
and some methods may miss the residual gate. Every attempt is saved with its actual status.
The DG companion remains experimental for shock-containing cone flow; smooth order does not establish shock robustness.


In [ ]:
Physics = "inviscid" # @param ["inviscid", "adiabatic", "isothermal", "dg-inviscid"]
Flux = "hllc" # @param ["roe", "vanleer", "hllc", "hlle", "rusanov", "ausm", "ausm-plus", "ausm-up", "ausm-up2", "slau2", "ec-lf"]
Use_default_physics_case = True # @param {type:"boolean"}
Cells = 60 # @param {type:"integer"}
Mach = 2.35 # @param {type:"number"}
Outer_angle_deg = 45.0 # @param {type:"number"}
Reconstruction = "cweno3" # @param ["first", "muscl", "cweno3", "cweno5", "cweno3-char", "cweno5-char", "primitive-minmod"]
DG_degree = 1 # @param [0, 1, 2]
Viscous_gradient_order = 2 # @param [2, 4]
Reynolds = 420000.0 # @param {type:"number"}
Prandtl = 0.72 # @param {type:"number"}
Wall_temperature_ratio = 1.0 # @param {type:"number"}
Max_SSPRK_steps = 12000 # @param {type:"integer"}
Max_Newton_attempts = 180 # @param {type:"integer"}
Use_matching_viscous_seed = True # @param {type:"boolean"}
Compare_all_fluxes = False # @param {type:"boolean"}
viscous_case = Physics in ("adiabatic", "isothermal")
actual_mach = (7.95 if viscous_case else 2.35) if Use_default_physics_case else Mach
actual_outer = (22.0 if viscous_case else 45.0) if Use_default_physics_case else Outer_angle_deg
actual_reconstruction = ("primitive-minmod" if viscous_case else "cweno3") if Use_default_physics_case else Reconstruction
selected_fluxes = list(FLUXES) if Compare_all_fluxes else [Flux]
OUT = WORK / "student-results"
results = []
for selected in selected_fluxes:
    result = run_case(flux=selected, physics=Physics, cells=Cells, mach=actual_mach,
        outer_deg=actual_outer, reconstruction=actual_reconstruction, degree=int(DG_degree),
        gradient_order=int(Viscous_gradient_order), reynolds=Reynolds, prandtl=Prandtl,
        wall_temperature_ratio=Wall_temperature_ratio, max_steps=Max_SSPRK_steps,
        max_newton=Max_Newton_attempts, use_seed=Use_matching_viscous_seed, output_dir=OUT)
    results.append(result)


## Physical properties and convergence

Solid curves met the recorded acceptance gate; dashed curves did not. Euler and DG use an independent
Taylor–Maccoll reference. Viscous runs use their own steady residual and explicit-drift gates;
Taylor–Maccoll is not their physical validation target. Temperature follows the perfect-gas equation.


In [ ]:
plot_comparison(results, OUT)
for r in results:
    print(json.dumps({k:r[k] for k in ("converged", "wall_pressure_ratio", "residual_rms_max_equation")}, indent=2))
    if "skin_friction_coefficient" in r:
        print("Cf =", r["skin_friction_coefficient"], "q_wall into fluid =", r["wall_heat_flux_into_fluid"])


## Export your run

The ZIP contains the generated PNG/PDF comparison, raw NPZ fields, modal coefficients for DG and
JSON configurations/statuses. The numerical modules remain visible in the notebook's working directory.
The full published study includes CSV profiles and vector figures. No results are uploaded automatically.


In [ ]:
import shutil
archive = shutil.make_archive(str(WORK / "my-conical-results"), "zip", OUT)
print("Saved:", archive)
try:
    from google.colab import files
except ImportError:
    print("Open the ZIP at the saved path, or download it from your notebook file browser.")
else:
    files.download(archive)


## Read and modify the algorithms

- `flux.py`, `ausm.py`, `entropy_flux.py`: 11 selectable Euler face fluxes.
- `reconstruction.py`, `fv_mesh.py`, `characteristics.py`: conservative FV reconstruction and nonuniform diffusion.
- `solver.py`: angular Euler finite volumes and SSPRK3.
- `viscous.py`, `steady.py`: local-station viscous equations and damped pseudo-transient Newton.
- `dg.py`, `verify_dg.py`: independent angular modal DG and a tested linear FR/Radau connection.
- `reference.py`: independent Taylor–Maccoll shooting reference.

References: [CWENO](https://arxiv.org/abs/1607.07319),
[AUSM+-up](https://doi.org/10.1016/j.jcp.2005.09.020),
[AUSM+-up2/SLAU2](https://doi.org/10.1016/j.jcp.2013.02.046),
[entropy-conservative flux](https://arxiv.org/abs/1209.4994),
[Cockburn and Shu RKDG](https://helper.ipam.ucla.edu/publications/pcatut2005/pcatut_5477_reprint2.pdf),
[NASA FR overview](https://ntrs.nasa.gov/citations/20160010066).

These are established research methods newly implemented in the course project, not methods invented in 2026.
The archive's original Fortran source files remain unchanged.
